# Generate data for project 2
I will take some audio segments from wav files created from maestro midi and turn them into spectrograms.

## 1. Load

In [3]:
import yaml
import pandas as pd
from midi2audio import FluidSynth
import pretty_midi
import os
import random
import numpy as np

In [4]:
import sys

sys.path.append(os.path.abspath(".."))

import utils.audio_wrapper as ap

In [5]:
with open("../config/data_config.yaml", "r") as f:
    config = yaml.safe_load(f)

timbres = config["timbres"]
soundfont_num = config["soundfont_num"]

maestro_data_dir = config["maestro_data_dir"]
maestro_metadata_file = config["maestro_metadata_file"]
wav_dir = config["wav_dir"]
spectrogram_dir = config["spectrogram_dir"]

num_images_to_create_per_timbre = config["num_images_to_create_per_timbre"]

In [6]:
maestro_metadata = pd.read_csv(maestro_metadata_file)

spectrogram_metadata = pd.DataFrame(columns=["maestro_id", "slice", "timbres", "spectrogram_path", "subset"])
try:
    spectrogram_metadata = pd.read_csv(f"{spectrogram_dir}/spectrogram_metadata.csv")
except FileNotFoundError:
    print("No existing spectrogram metadata found. A new empty dataframe is created.")

No existing spectrogram metadata found. A new empty dataframe is created.


# 2. Choosing midi files to create wav

In [7]:
choosed_subset = "test" # will change to "validation" and "test" later
subset_metadata = maestro_metadata[maestro_metadata["split"] == choosed_subset]

required_total_audio_length = num_images_to_create_per_timbre[choosed_subset] * 7 # about 6 seconds for each image, plus 1 for safety


used_pairs = set(zip(spectrogram_metadata["maestro_id"], spectrogram_metadata["timbres"]))

available_midi_ids = {
    timbre: [
        maestro_id for maestro_id in range(len(subset_metadata))
        if (maestro_id, timbre) not in used_pairs
    ]
    for timbre in timbres
} # I will not use all of these

selected_midi_ids = { timbre: [] for timbre in timbres }

# for each timbre, choose the songs of which combined duration is at least required_total_audio_length
for timbre in timbres:
    temp = available_midi_ids[timbre]
    random.Random(42).shuffle(temp)
    total_audio_length = 0
    while total_audio_length < required_total_audio_length:
        id = temp.pop(0) # take a random available midi id
        duration = subset_metadata.iloc[id]["duration"]
        total_audio_length += duration
        selected_midi_ids[timbre].append(id)
        
selected_midi_ids

{'piano': [148, 36], 'flute': [148, 36]}

## 3. Create midi file then create spectrogram

In [8]:
soundfont_path = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
fs = FluidSynth(sound_font=soundfont_path)

def create_wav_file_from_midi(midi_file_path, output_path, timbre):
    temp_midi_file_path = f'{wav_dir}/temp_song.mid'
    midi_data = pretty_midi.PrettyMIDI(midi_file_path)
    
    for instrument in midi_data.instruments: # turn all instruments to the same timbre for consistency
        instrument.program = soundfont_num[timbre]
    midi_data.write(temp_midi_file_path)
    fs.midi_to_audio(temp_midi_file_path, output_path)
    print(f"Created {output_path}")
    os.remove(temp_midi_file_path)


num_of_created_spectrograms = 0

# for each (timbre, id), create wav file, then create spectrograms
for timbre in timbres:
    for id in selected_midi_ids[timbre]:
        midi_file_path = f'{maestro_data_dir}/{maestro_metadata.iloc[id]["midi_filename"]}'
        audio_path = f'{wav_dir}/song_{id}_{timbre}.wav'
        create_wav_file_from_midi(midi_file_path=midi_file_path, output_path=audio_path, timbre=timbre)

        audio_wrapper = ap.AudioWrapper()
        audio_wrapper.load_audio(audio_path)

        start, end = 0, audio_wrapper.number_of_slices - 1 # we will add slice[start:end] to the dataset
        if audio_wrapper.number_of_slices + num_of_created_spectrograms > num_images_to_create_per_timbre[choosed_subset]:
            remaining_needed_image = audio_wrapper.number_of_slices + num_of_created_spectrograms - num_images_to_create_per_timbre[choosed_subset]
            end = end - remaining_needed_image # garantee num_images_to_create_per_timbre[choosed_subset] 
        num_of_created_spectrograms += (end - start + 1)

        for i in range(start, end + 1):
            save_path = f'{spectrogram_dir}/spectrogram_{id}_slice_{i}_{timbre}.npy'
            spectrogram = audio_wrapper.spectrogram_from_audio_slice(i)
            audio_wrapper.save_spectrogram(spectrogram, save_path)
            print(f"Created {save_path}")

            #add to metadata
            spectrogram_metadata.loc[len(spectrogram_metadata)] = {
            "maestro_id": id,
            "slice": i,
            "timbres": timbre,
            "spectrogram_path": save_path,
            "subset": choosed_subset
        }


[W][09:45:40.438960] pw.conf      | [          conf.c: 1182 try_load_conf()] can't load config client.conf: No such file or directory
[E][09:45:40.439080] pw.conf      | [          conf.c: 1215 pw_conf_load_conf_for_context()] can't load config client.conf: No such file or directory

(process:6958): GLib-GObject-CRITICAL **: 09:45:40.446: g_param_spec_enum: assertion 'g_enum_get_value (enum_class, default_value) != NULL' failed

(process:6958): GLib-GObject-CRITICAL **: 09:45:40.446: validate_pspec_to_install: assertion 'G_IS_PARAM_SPEC (pspec)' failed

(process:6958): GLib-GObject-CRITICAL **: 09:45:40.446: g_param_spec_ref_sink: assertion 'G_IS_PARAM_SPEC (pspec)' failed

(process:6958): GLib-GObject-CRITICAL **: 09:45:40.446: g_param_spec_unref: assertion 'G_IS_PARAM_SPEC (pspec)' failed


FluidSynth runtime version 2.4.8
Copyright (C) 2000-2025 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '/home/vtqn/projects/project2/data/wav/song_148_piano.wav'..
Created /home/vtqn/projects/project2/data/wav/song_148_piano.wav


/home/vtqn/projects/project2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_0_piano.npy
Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_1_piano.npy
Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_2_piano.npy
Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_3_piano.npy
Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_4_piano.npy
Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_5_piano.npy
Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_6_piano.npy
Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_7_piano.npy
Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_8_piano.npy
Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_9_piano.npy
Created /home/vtqn/projects/project2/data/spectrogram/spectrogram_148_slice_10_piano.npy
Created /home/vtqn/projects/pro

[W][09:46:05.471781] pw.conf      | [          conf.c: 1182 try_load_conf()] can't load config client.conf: No such file or directory
[E][09:46:05.471897] pw.conf      | [          conf.c: 1215 pw_conf_load_conf_for_context()] can't load config client.conf: No such file or directory

(process:7071): GLib-GObject-CRITICAL **: 09:46:05.475: g_param_spec_enum: assertion 'g_enum_get_value (enum_class, default_value) != NULL' failed

(process:7071): GLib-GObject-CRITICAL **: 09:46:05.475: validate_pspec_to_install: assertion 'G_IS_PARAM_SPEC (pspec)' failed

(process:7071): GLib-GObject-CRITICAL **: 09:46:05.475: g_param_spec_ref_sink: assertion 'G_IS_PARAM_SPEC (pspec)' failed

(process:7071): GLib-GObject-CRITICAL **: 09:46:05.475: g_param_spec_unref: assertion 'G_IS_PARAM_SPEC (pspec)' failed


FluidSynth runtime version 2.4.8
Copyright (C) 2000-2025 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '/home/vtqn/projects/project2/data/wav/song_36_piano.wav'..
Created /home/vtqn/projects/project2/data/wav/song_36_piano.wav


[W][09:46:11.401767] pw.conf      | [          conf.c: 1182 try_load_conf()] can't load config client.conf: No such file or directory
[E][09:46:11.401903] pw.conf      | [          conf.c: 1215 pw_conf_load_conf_for_context()] can't load config client.conf: No such file or directory

(process:7101): GLib-GObject-CRITICAL **: 09:46:11.405: g_param_spec_enum: assertion 'g_enum_get_value (enum_class, default_value) != NULL' failed

(process:7101): GLib-GObject-CRITICAL **: 09:46:11.405: validate_pspec_to_install: assertion 'G_IS_PARAM_SPEC (pspec)' failed

(process:7101): GLib-GObject-CRITICAL **: 09:46:11.405: g_param_spec_ref_sink: assertion 'G_IS_PARAM_SPEC (pspec)' failed

(process:7101): GLib-GObject-CRITICAL **: 09:46:11.405: g_param_spec_unref: assertion 'G_IS_PARAM_SPEC (pspec)' failed


FluidSynth runtime version 2.4.8
Copyright (C) 2000-2025 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '/home/vtqn/projects/project2/data/wav/song_148_flute.wav'..
Created /home/vtqn/projects/project2/data/wav/song_148_flute.wav


[W][09:46:20.547524] pw.conf      | [          conf.c: 1182 try_load_conf()] can't load config client.conf: No such file or directory
[E][09:46:20.547637] pw.conf      | [          conf.c: 1215 pw_conf_load_conf_for_context()] can't load config client.conf: No such file or directory

(process:7149): GLib-GObject-CRITICAL **: 09:46:20.551: g_param_spec_enum: assertion 'g_enum_get_value (enum_class, default_value) != NULL' failed

(process:7149): GLib-GObject-CRITICAL **: 09:46:20.551: validate_pspec_to_install: assertion 'G_IS_PARAM_SPEC (pspec)' failed

(process:7149): GLib-GObject-CRITICAL **: 09:46:20.551: g_param_spec_ref_sink: assertion 'G_IS_PARAM_SPEC (pspec)' failed

(process:7149): GLib-GObject-CRITICAL **: 09:46:20.551: g_param_spec_unref: assertion 'G_IS_PARAM_SPEC (pspec)' failed


FluidSynth runtime version 2.4.8
Copyright (C) 2000-2025 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '/home/vtqn/projects/project2/data/wav/song_36_flute.wav'..
Created /home/vtqn/projects/project2/data/wav/song_36_flute.wav


In [9]:
spectrogram_metadata.head()

,maestro_id,slice,timbres,spectrogram_path,subset
0,148,0,piano,/home/vtqn/projects/project2/data/spectrogram/...,test
1,148,1,piano,/home/vtqn/projects/project2/data/spectrogram/...,test
2,148,2,piano,/home/vtqn/projects/project2/data/spectrogram/...,test
3,148,3,piano,/home/vtqn/projects/project2/data/spectrogram/...,test
4,148,4,piano,/home/vtqn/projects/project2/data/spectrogram/...,test


In [10]:
spectrogram_metadata.describe()

,maestro_id,slice
count,100.0,100.000000
mean,148.0,49.500000
std,0.0,29.011492
min,148.0,0.000000
25%,148.0,24.750000
50%,148.0,49.500000
75%,148.0,74.250000
max,148.0,99.000000


In [11]:
len(spectrogram_metadata)

100

In [12]:
spectrogram_metadata.to_csv(f"{spectrogram_dir}/spectrogram_metadata.csv", index=False)

In [13]:
# wav is not neccessary to keep anymore
!rm -rf {wav_dir}
!mkdir -p {wav_dir}

In [ ]:
# # in the case everything above is not good
# !rm -rf {spectrogram_dir}
# !mkdir -p {spectrogram_dir}

In [15]:
sample_spectrogram = np.load(spectrogram_metadata.iloc[3]["spectrogram_path"])
print(sample_spectrogram.shape)

(512, 512)


In [ ]:
# experiment
import sys
import os
import numpy as np

sys.path.append(os.path.abspath(".."))

import utils.audio_wrapper as ap

hop_length = ap.SpectrogramConverter().hop_length
nfft = ap.SpectrogramConverter().n_fft
image_width = ap.SpectrogramConverter().image_width
win_length = ap.SpectrogramConverter().win_length

N = (image_width) * hop_length - 1

audio = np.random.randn(N).astype(np.float32)

aw = ap.AudioWrapper()
aw.audio = audio
aw.converter = ap.SpectrogramConverter(
    image_width=512,
    n_mels=512,
    sample_rate=44100,
    n_fft=17640,
    hop_length=441,
    win_length=4410
)
aw.slice_size = (aw.converter.image_width) * aw.converter.hop_length # - aw.converter.win_length + aw.converter.n_fft
aw.number_of_slices = len(aw.audio) // aw.slice_size

audio_slice = aw.get_audio_slices(0)
print(audio_slice.shape)
spectrogram = aw.converter.spectrogram_from_waveform(audio_slice)
print(spectrogram.shape)

(225791,)
(512, 512)
